In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../Dataset/clean_amazon_store_sales.csv")

df["Order Date"] = pd.to_datetime(df["Order Date"])
df["Ship Date"] = pd.to_datetime(df["Ship Date"])

In [2]:
total_sales = df["Sales"].sum()
total_profit = df["Profit"].sum()
total_orders = df["Order ID"].nunique()
total_customers = df["Customer ID"].nunique()
total_quantity = df["Quantity"].sum()

returned_orders = df.loc[df["Returns"] == 1, "Order ID"].nunique()

profit_margin = (total_profit / total_sales) * 100
return_rate = (returned_orders / total_orders) * 100
average_order_value = total_sales / total_orders
average_shipping_days = df["Shipping Days"].mean()

kpi_summary = pd.DataFrame({
    "KPI": [
        "Total Sales",
        "Total Profit",
        "Profit Margin %",
        "Total Orders",
        "Total Customers",
        "Total Quantity",
        "Returned Orders",
        "Return Rate %",
        "Average Order Value",
        "Average Shipping Days"
    ],
    "Value": [
        round(total_sales, 2),
        round(total_profit, 2),
        round(profit_margin, 2),
        total_orders,
        total_customers,
        total_quantity,
        returned_orders,
        round(return_rate, 2),
        round(average_order_value, 2),
        round(average_shipping_days, 2)
    ]
})

kpi_summary

,KPI,Value
0,Total Sales,1565804.32
1,Total Profit,175262.11
2,Profit Margin %,11.19
3,Total Orders,3003.00
4,Total Customers,773.00
5,Total Quantity,22317.00
6,Returned Orders,104.00
7,Return Rate %,3.46
8,Average Order Value,521.41
9,Average Shipping Days,3.93


In [3]:
category_summary = (
    df.groupby("Category")
      .agg(
          Total_Sales=("Sales", "sum"),
          Total_Profit=("Profit", "sum"),
          Total_Orders=("Order ID", "nunique")
      )
      .sort_values("Total_Sales", ascending=False)
)

category_summary

,Total_Sales,Total_Profit,Total_Orders
Category,,,
Office Supplies,643707.6870,74797.2461,2219
Technology,470587.9910,90458.2486,915
Furniture,451508.6452,10006.6112,1040


In [4]:
top_sales_category = category_summary["Total_Sales"].idxmax()
top_profit_category = category_summary["Total_Profit"].idxmax()

print("Top category by sales:", top_sales_category)
print("Top category by profit:", top_profit_category)

Top category by sales: Office Supplies
Top category by profit: Technology


In [5]:
subcategory_summary = (
    df.groupby(["Category", "Sub-Category"])
      .agg(
          Total_Sales=("Sales", "sum"),
          Total_Profit=("Profit", "sum")
      )
      .sort_values("Total_Profit")
)

loss_making_subcategories = subcategory_summary[
    subcategory_summary["Total_Profit"] < 0
]

loss_making_subcategories

,,Total_Sales,Total_Profit
Category,Sub-Category,,
Furniture,Tables,119293.7430,-11091.6365
Office Supplies,Supplies,36720.9860,-1654.2767
Furniture,Bookcases,57577.6862,-342.8883


In [6]:
region_summary = (
    df.groupby("Region")
      .agg(
          Total_Sales=("Sales", "sum"),
          Total_Profit=("Profit", "sum"),
          Total_Orders=("Order ID", "nunique")
      )
      .sort_values("Total_Profit", ascending=False)
)

region_summary

,Total_Sales,Total_Profit,Total_Orders
Region,,,
West,522441.0520,67859.9582,961
East,450234.6660,53400.4243,844
Central,341007.5242,27450.0071,711
South,252121.0810,26551.7163,487


In [7]:
best_region = region_summary["Total_Profit"].idxmax()
lowest_region = region_summary["Total_Profit"].idxmin()

print("Best region by profit:", best_region)
print("Lowest region by profit:", lowest_region)

Best region by profit: West
Lowest region by profit: South


In [8]:
segment_summary = (
    df.groupby("Segment")
      .agg(
          Total_Sales=("Sales", "sum"),
          Total_Profit=("Profit", "sum"),
          Customers=("Customer ID", "nunique"),
          Orders=("Order ID", "nunique")
      )
      .sort_values("Total_Sales", ascending=False)
)

segment_summary

,Total_Sales,Total_Profit,Customers,Orders
Segment,,,,
Consumer,753002.1291,81338.5875,400,1528
Corporate,509743.1262,57805.7991,230,915
Home Office,303059.0679,36117.7193,143,560


In [9]:
return_category = (
    df.groupby("Category")
      .agg(
          Total_Orders=("Order ID", "nunique"),
          Returned_Orders=("Returns", lambda x: (x == 1).sum())
      )
)

return_category["Return Rate %"] = (
    return_category["Returned_Orders"] /
    return_category["Total_Orders"] * 100
).round(2)

return_category.sort_values("Return Rate %", ascending=False)

,Total_Orders,Returned_Orders,Return Rate %
Category,,,
Office Supplies,2219,165,7.44
Furniture,1040,70,6.73
Technology,915,52,5.68


In [10]:
return_shipping = (
    df.groupby("Ship Mode")
      .agg(
          Total_Orders=("Order ID", "nunique"),
          Returned_Orders=("Returns", lambda x: (x == 1).sum())
      )
)

return_shipping["Return Rate %"] = (
    return_shipping["Returned_Orders"] /
    return_shipping["Total_Orders"] * 100
).round(2)

return_shipping.sort_values("Return Rate %", ascending=False)

,Total_Orders,Returned_Orders,Return Rate %
Ship Mode,,,
First Class,499,66,13.23
Second Class,568,57,10.04
Standard Class,1773,154,8.69
Same Day,163,10,6.13


In [11]:
product_summary = (
    df.groupby("Product Name")
      .agg(
          Total_Sales=("Sales", "sum"),
          Total_Profit=("Profit", "sum"),
          Quantity_Sold=("Quantity", "sum")
      )
)

top_10_products = product_summary.sort_values(
    "Total_Sales", ascending=False
).head(10)

bottom_10_products = product_summary.sort_values(
    "Total_Profit", ascending=True
).head(10)

top_10_products

,Total_Sales,Total_Profit,Quantity_Sold
Product Name,,,
"3D Systems Cube Printer, 2nd Generation, Magenta",14334.890,3717.9714,11
Canon imageCLASS 2200 Advanced Copier,14076.824,25199.9280,20
Hewlett Packard LaserJet 3310 Copier,13837.732,6407.8932,31
GBC DocuBind TL300 Electric Binding System,12890.258,2753.7593,21
GBC DocuBind P400 Electric Binding System,12577.108,762.1544,16
Samsung Galaxy Mega 6.3,12370.708,1696.7596,34
Martin Yale Chadless Opener Electric Letter Opener,12268.902,-1232.5588,16
HON 5400 Series Task Chairs for Big and Tall,11887.562,70.0980,21
Global Troy Executive Leather Low-Back Tilter,10217.894,776.5190,25


In [12]:
bottom_10_products

,Total_Sales,Total_Profit,Quantity_Sold
Product Name,,,
Cubify CubeX 3D Printer Double Head Print,9323.969,-6239.9792,7
Cubify CubeX 3D Printer Triple Head Print,3009.980,-3839.9904,4
Lexmark MX611dhe Monochrome Laser Printer,5676.967,-2719.9840,7
Bush Advantage Collection Racetrack Conference Table,4334.100,-2019.2396,18
Ibico EPK-21 Electric Binding System,6437.966,-1285.1932,8
Martin Yale Chadless Opener Electric Letter Opener,12268.902,-1232.5588,16
BoxOffice By Design Rectangular and Half-Moon Meeting Room Tables,1052.500,-986.5625,13
Bretford “Just In Time” Height-Adjustable Multi-Task Work Tables,3869.900,-964.1940,17
Zebra GK420t Direct Thermal/Thermal Transfer Printer,703.710,-938.2800,6


In [13]:
payment_summary = (
    df.groupby("Payment Mode")
      .agg(
          Total_Sales=("Sales", "sum"),
          Total_Profit=("Profit", "sum"),
          Orders=("Order ID", "nunique")
      )
      .sort_values("Total_Sales", ascending=False)
)

payment_summary

,Total_Sales,Total_Profit,Orders
Payment Mode,,,
COD,667417.7513,82092.4250,1717
Online,553993.4607,54047.5234,1562
Cards,344393.1112,39122.1575,985


In [14]:
shipping_summary = (
    df.groupby("Ship Mode")
      .agg(
          Average_Shipping_Days=("Shipping Days", "mean"),
          Total_Sales=("Sales", "sum"),
          Total_Profit=("Profit", "sum")
      )
      .sort_values("Average_Shipping_Days")
)

shipping_summary

,Average_Shipping_Days,Total_Sales,Total_Profit
Ship Mode,,,
Same Day,0.052326,95958.5010,8808.8243
First Class,2.140772,242936.7194,29749.8665
Second Class,3.217088,314508.0640,36936.0265
Standard Class,5.045494,912401.0388,99767.3886


In [15]:
monthly_summary = (
    df.groupby(df["Order Date"].dt.to_period("M"))
      .agg(
          Total_Sales=("Sales", "sum"),
          Total_Profit=("Profit", "sum"),
          Total_Orders=("Order ID", "nunique")
      )
      .reset_index()
)

monthly_summary["Order Date"] = monthly_summary["Order Date"].astype(str)

monthly_summary

,Order Date,Total_Sales,Total_Profit,Total_Orders
0,2019-01,18616.4310,2853.0901,48
1,2019-02,19978.8150,5004.5795,45
2,2019-03,51715.8750,3611.9680,86
3,2019-04,38750.0390,2977.8149,89
4,2019-05,50987.7280,8662.1464,108
5,2019-06,40344.5340,4750.3781,97
6,2019-07,39261.9630,4432.8779,96
7,2019-08,31115.3743,2062.0693,90
8,2019-09,73410.0249,9328.6576,192
9,2019-10,42687.7450,16243.1425,105


In [16]:
best_sales_month = monthly_summary.loc[
    monthly_summary["Total_Sales"].idxmax()
]

best_profit_month = monthly_summary.loc[
    monthly_summary["Total_Profit"].idxmax()
]

print("Best sales month:")
print(best_sales_month)

print("\nBest profit month:")
print(best_profit_month)

Best sales month:
Order Date          2020-12
Total_Sales     166185.8488
Total_Profit       8482.742
Total_Orders            225
Name: 23, dtype: object

Best profit month:
Order Date         2019-12
Total_Sales      78399.043
Total_Profit    17885.3093
Total_Orders           176
Name: 11, dtype: object


# Executive Business Insights

## 1. Overall Performance

The business generated total sales of **[Total Sales]** and total profit of
**[Total Profit]**, resulting in a profit margin of **[Profit Margin %]**.

A total of **[Total Orders]** unique orders were placed by **[Total Customers]**
customers. The average order value was **[Average Order Value]**.

## 2. Category Performance

**[Top Sales Category]** generated the highest sales, while
**[Top Profit Category]** generated the highest profit.

Loss-making sub-categories, including **[Loss-Making Sub-Category]**, should be
reviewed for discounting, shipping cost, supplier cost, or pricing issues.

## 3. Regional Performance

**[Best Region]** was the strongest region by profitability.

**[Lowest Region]** had the lowest profit performance and should be investigated
for high discounts, high returns, poor product mix, or fulfillment costs.

## 4. Customer Segment Performance

The **[Top Customer Segment]** segment contributed the highest sales and should
be prioritized through retention campaigns, targeted offers, and personalized
product recommendations.

## 5. Returns and Shipping

The overall return rate was **[Return Rate %]**.

**[Highest Return Category]** had the highest return rate by category, while
**[Highest Return Shipping Mode]** had the highest return rate by shipping mode.

This may indicate product-quality, expectation-setting, packaging, or delivery
experience issues.

## 6. Product Performance

The highest-selling product was **[Top Product]**.

The lowest-profit product was **[Lowest Profit Product]**. Products with
negative profit should be reviewed before continuing promotions or discounts.

## 7. Monthly Trend

The strongest sales month was **[Best Sales Month]**.

The strongest profit month was **[Best Profit Month]**. Management should study
the products, regions, campaigns, and customer segments responsible for these
peak periods.

# Business Recommendations

1. Review loss-making sub-categories and products to identify whether discounts,
shipping costs, supplier costs, or pricing are causing negative profit.

2. Prioritize high-profit categories and regions in marketing, inventory planning,
and promotional campaigns.

3. Investigate categories and shipping modes with high return rates to improve
product descriptions, packaging, delivery quality, and customer expectations.

4. Focus retention campaigns on the highest-value customer segment and customers
with high purchase frequency.

5. Use monthly sales and profit trends to plan inventory, staffing, promotions,
and regional sales targets.

6. Monitor average shipping days by shipping mode and improve slow delivery
channels where they affect returns or customer satisfaction.

7. Build the Power BI dashboard using these KPIs and insights so management can
track performance interactively.

In [17]:
kpi_summary.to_csv("../Documentation/kpi_summary.csv", index=False)
category_summary.to_csv("../Documentation/category_summary.csv")
region_summary.to_csv("../Documentation/region_summary.csv")
segment_summary.to_csv("../Documentation/segment_summary.csv")
return_category.to_csv("../Documentation/return_by_category.csv")
return_shipping.to_csv("../Documentation/return_by_shipping_mode.csv")
top_10_products.to_csv("../Documentation/top_10_products.csv")
bottom_10_products.to_csv("../Documentation/bottom_10_products.csv")